# Qwen3-TTS 0.6B：Google Colab 生成 10 条英语 WAV

此 Notebook 使用：
- `Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice`
- `float16`
- 优先使用 `sdpa`，失败时自动改用 `eager`
- 每批 2 条
- 输出 WAV 到 Google Drive
- 支持已完成跳过、进度记录和失败重试

运行前，在 Colab 中选择：**运行时 → 更改运行时类型 → GPU**。

In [ ]:
# 1. 检查 GPU
import subprocess
import torch

if not torch.cuda.is_available():
    raise RuntimeError('没有检测到 GPU。请在 Colab 中选择：运行时 → 更改运行时类型 → GPU，然后重新运行。')

print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA:', torch.version.cuda)
subprocess.run(['nvidia-smi'], check=False)


In [ ]:
# 2. 安装官方 Qwen3-TTS Python 包
%pip install -q -U qwen-tts soundfile
print('安装完成。')


In [ ]:
# 3. 挂载 Google Drive，并创建输出目录
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

OUTPUT_DIR = Path('/content/drive/MyDrive/qwen3_tts_test_10')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('输出目录:', OUTPUT_DIR)


In [ ]:
# 4. 准备 10 条英语句子
# 后续可把这里替换成你自己的句子，或改成从 CSV/TXT 读取。

sentences = [
    'Good morning. I hope you have a wonderful day.',
    'The weather is beautiful, so let us take a walk outside.',
    'Please read each sentence clearly and naturally.',
    'Learning English takes patience, practice, and confidence.',
    'She arrived at the station five minutes before the train left.',
    'Could you tell me where the nearest library is?',
    'Technology has changed the way people work and communicate.',
    'He paused for a moment before answering the difficult question.',
    'Every small improvement brings us closer to our goal.',
    'Thank you for listening, and I will see you next time.'
]

records = [
    {'id': index + 1, 'text': text}
    for index, text in enumerate(sentences)
]

print('句子数量:', len(records))
for row in records:
    print(f"{row['id']:05d}: {row['text']}")


In [ ]:
# 5. 加载模型
# T4 等免费 Colab GPU 优先用 float16 + sdpa，不安装 FlashAttention。

import gc
import torch
from qwen_tts import Qwen3TTSModel

MODEL_ID = 'Qwen/Qwen3-TTS-12Hz-0.6B-CustomVoice'

def load_model():
    errors = []
    for attention in ('sdpa', 'eager'):
        try:
            print(f'正在加载模型，attention={attention} ...')
            loaded_model = Qwen3TTSModel.from_pretrained(
                MODEL_ID,
                device_map='cuda:0',
                dtype=torch.float16,
                attn_implementation=attention,
            )
            print(f'模型加载成功，attention={attention}')
            return loaded_model, attention
        except Exception as exc:
            errors.append(f'{attention}: {type(exc).__name__}: {exc}')
            print(f'{attention} 加载失败，准备尝试下一种模式。')
            gc.collect()
            torch.cuda.empty_cache()

    raise RuntimeError('模型加载失败：\n' + '\n'.join(errors))

model, attention_mode = load_model()

print('支持的语言:', model.get_supported_languages())
print('支持的音色:', model.get_supported_speakers())


In [ ]:
# 6. 批量生成 WAV：断点续跑、错误重试、已完成跳过

import csv
import json
import time
import traceback
import soundfile as sf

SPEAKER = 'Aiden'  # 英语男声；也可以改为 Ryan
INSTRUCT = 'Speak clearly and naturally at a moderate pace.'
BATCH_SIZE = 2
MAX_RETRIES = 3
PROGRESS_FILE = OUTPUT_DIR / 'progress.json'
MANIFEST_FILE = OUTPUT_DIR / 'manifest.csv'

def output_path(item):
    return OUTPUT_DIR / f"{item['id']:05d}.wav"

def save_progress(completed_ids, failures):
    payload = {
        'model': MODEL_ID,
        'speaker': SPEAKER,
        'attention': attention_mode,
        'completed_ids': sorted(completed_ids),
        'failures': failures,
        'updated_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    }
    PROGRESS_FILE.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding='utf-8'
    )

def generate(items):
    return model.generate_custom_voice(
        text=[item['text'] for item in items],
        language=['English'] * len(items),
        speaker=[SPEAKER] * len(items),
        instruct=[INSTRUCT] * len(items),
    )

def generate_with_retry(items):
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            print(
                f"生成 {[item['id'] for item in items]}，"
                f"第 {attempt}/{MAX_RETRIES} 次尝试"
            )
            return generate(items)
        except Exception as exc:
            last_error = exc
            print(f'生成失败: {type(exc).__name__}: {exc}')
            gc.collect()
            torch.cuda.empty_cache()
            time.sleep(attempt * 2)
    raise last_error

completed_ids = {
    item['id'] for item in records if output_path(item).exists()
}
failures = {}

pending = [item for item in records if item['id'] not in completed_ids]
print(f'已完成 {len(completed_ids)} 条，待生成 {len(pending)} 条。')

for start in range(0, len(pending), BATCH_SIZE):
    batch = pending[start:start + BATCH_SIZE]

    try:
        wavs, sample_rate = generate_with_retry(batch)
        for item, wav in zip(batch, wavs):
            target = output_path(item)
            sf.write(str(target), wav, sample_rate)
            completed_ids.add(item['id'])
            failures.pop(str(item['id']), None)
            print('已保存:', target.name)

    except Exception as batch_error:
        print('整批生成失败，改为逐条生成。')
        print(traceback.format_exc(limit=1))

        for item in batch:
            if output_path(item).exists():
                completed_ids.add(item['id'])
                continue

            try:
                wavs, sample_rate = generate_with_retry([item])
                sf.write(str(output_path(item)), wavs[0], sample_rate)
                completed_ids.add(item['id'])
                failures.pop(str(item['id']), None)
                print('逐条生成成功:', output_path(item).name)
            except Exception as single_error:
                failures[str(item['id'])] = {
                    'text': item['text'],
                    'error': f'{type(single_error).__name__}: {single_error}',
                }
                print(f"第 {item['id']} 条最终失败，已记录，继续执行后面的句子。")

    save_progress(completed_ids, failures)

# 保存清单
with MANIFEST_FILE.open('w', newline='', encoding='utf-8-sig') as file:
    writer = csv.DictWriter(
        file,
        fieldnames=['id', 'filename', 'text', 'status']
    )
    writer.writeheader()
    for item in records:
        writer.writerow({
            'id': item['id'],
            'filename': output_path(item).name,
            'text': item['text'],
            'status': 'completed' if output_path(item).exists() else 'failed',
        })

print('\n生成结束。')
print('成功:', len(completed_ids))
print('失败:', len(failures))
print('目录:', OUTPUT_DIR)


In [ ]:
# 7. 在 Colab 中试听生成结果
from IPython.display import Audio, Markdown, display

wav_files = sorted(OUTPUT_DIR.glob('*.wav'))
print('找到 WAV 文件:', len(wav_files))

for wav_file in wav_files[:10]:
    display(Markdown(f'**{wav_file.name}**'))
    display(Audio(filename=str(wav_file)))


In [ ]:
# 8. 可选：把本次结果打包成 ZIP，方便一次下载
import shutil
from IPython.display import FileLink, display

ZIP_BASE = '/content/qwen3_tts_test_10'
zip_path = shutil.make_archive(ZIP_BASE, 'zip', root_dir=str(OUTPUT_DIR))
print('ZIP:', zip_path)
display(FileLink(zip_path))


## 后续换成自己的句子

只需要修改第 4 格中的 `sentences`。文件名由句子编号生成，例如：

- `00001.wav`
- `00002.wav`
- `00010.wav`

重新运行生成格时，已经存在的 WAV 会自动跳过。